### database connection and read data from parquet file

In [ ]:
import pandas as pd
import pandas as pd
from sqlalchemy import create_engine, text

# 数据库配置
username = "XXXXXX"
password = "YYYYYY"
host = "localhost"
port = 5432
database = "eyewear-data"

# 创建连接
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)


df = pd.read_parquet("customerinfo_cluster.parquet")
df


,customer_id,registration_date,total_orders,total_spend,avg_order_value,first_purchase_date,last_purchase_date,purchase_frequency,repeat_customer_flag,has_purchase,...,recency_days,total_items,frame_ratio,lens_ratio,ai_glasses_ratio,sunglasses_ratio,avg_discount_percentage,discount_usage_rate,promo_order_ratio,cluster
0,1,2018-09-18,2,581.45,290.725000,2023-07-04 15:29:48,2023-11-08 17:19:27,0.000817,1,1,...,570.0,3.0,1.000000,0.000000,0.000000,0.666667,0.120000,0.666667,0.500000,0
1,3,2017-07-20,1,434.35,434.350000,2023-07-21 00:27:06,2023-07-21 00:27:06,0.000348,0,1,...,680.0,2.0,1.000000,0.000000,0.500000,0.000000,0.000000,0.000000,1.000000,1
2,7,2024-10-11,2,923.72,461.860000,2024-02-07 15:40:29,2024-06-05 14:57:23,0.008584,1,1,...,359.0,4.0,1.000000,0.000000,0.500000,0.250000,0.000000,0.000000,0.500000,4
3,13,2023-01-08,2,612.46,306.230000,2023-09-13 09:17:44,2024-11-18 12:25:53,0.002286,1,1,...,194.0,2.0,1.000000,0.000000,0.000000,0.500000,0.000000,0.000000,0.000000,4
4,14,2019-06-06,6,2209.77,368.295000,2023-05-02 16:02:36,2024-04-30 13:25:49,0.002743,1,1,...,397.0,8.0,0.875000,0.125000,0.375000,0.125000,0.025000,0.125000,0.166667,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149994,159992,2024-03-10,6,2189.91,364.985000,2023-07-09 20:57:30,2024-07-27 21:22:33,0.013393,1,1,...,307.0,8.0,0.750000,0.250000,0.125000,0.250000,0.077500,0.625000,0.500000,3
149995,159994,2017-12-01,4,1736.47,434.117500,2023-06-24 15:58:27,2023-11-08 07:05:48,0.001460,1,1,...,571.0,7.0,0.714286,0.285714,0.428571,0.142857,0.085714,0.428571,0.250000,4
149996,159995,2023-01-18,9,5819.59,646.621111,2023-03-10 18:01:33,2024-11-20 12:24:46,0.010405,1,1,...,190.0,23.0,0.826087,0.173913,0.347826,0.217391,0.107647,0.411765,0.444444,3
149997,159997,2017-03-14,6,2188.18,364.696667,2023-06-17 11:36:20,2024-12-10 15:05:23,0.001999,1,1,...,169.0,9.0,1.000000,0.000000,0.222222,0.444444,0.042857,0.142857,0.166667,3


### insert customers' `cluster`

In [5]:

payload = df[["customer_id", "cluster"]].to_dict(orient="records")

with engine.begin() as conn:
    conn.execute(
        text("""
            UPDATE "CustomerInfo"
            SET customer_hierarchy = :cluster
            WHERE customer_id = :customer_id
        """),
        payload
    )

### insert `customer_type` by customers' `cluster`

In [10]:
mapping = {
    0: "Promotional Sensitive Customers",
    1: "One-time Customers",
    2: "Lens Customers",
    3: "VIP Loyal Customers",
    4: "Regular Customers",
}

df = df[df["cluster"].notna()].copy()
df["customer_type"] = df["cluster"].map(mapping)

payload2 = df[["customer_id", "customer_type"]].to_dict(orient="records")
# payload2

In [ ]:
with engine.begin() as conn:
    conn.execute(
        text("""
            UPDATE "CustomerInfo"
            SET customer_type = :customer_type
            WHERE customer_id = :customer_id
        """),
        payload2
    )